In [54]:
# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝 
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_absolute_error, mean_squared_error, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMRegressor

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [55]:
df = pd.read_csv('data/merged_final_data.csv')

In [56]:
cols = df.columns
cols 

Index(['order_id', 'customer_id', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'customer_lat', 'customer_lng', 'total_items_count', 'seller_id',
       'price', 'freight_value', 'review_score', 'category',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'seller_lat',
       'seller_lng', 'order_purchase_dayofweek', 'order_purchase_month',
       'approved_days', 'dispatch_days', 'delivery_days',
       'expected_delivery_days', 'delay_days', 'delay_days_int', 'is_delayed',
       'delay_days_cat', 'main_category', 'sub_category'],
      dtype='str')

In [57]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95784 entries, 0 to 95783
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  95784 non-null  str    
 1   customer_id               95784 non-null  str    
 2   customer_unique_id        95784 non-null  str    
 3   customer_zip_code_prefix  95784 non-null  int64  
 4   customer_city             95784 non-null  str    
 5   customer_state            95784 non-null  str    
 6   customer_lat              95735 non-null  float64
 7   customer_lng              95735 non-null  float64
 8   total_items_count         95784 non-null  int64  
 9   seller_id                 95784 non-null  str    
 10  price                     95784 non-null  float64
 11  freight_value             95784 non-null  float64
 12  review_score              95784 non-null  int64  
 13  category                  94414 non-null  str    
 14  seller_zip_code_p

In [58]:
df.isna().sum()

order_id                       0
customer_id                    0
customer_unique_id             0
customer_zip_code_prefix       0
customer_city                  0
customer_state                 0
customer_lat                  49
customer_lng                  49
total_items_count              0
seller_id                      0
price                          0
freight_value                  0
review_score                   0
category                    1370
seller_zip_code_prefix         0
seller_city                    0
seller_state                   0
seller_lat                     0
seller_lng                     0
order_purchase_dayofweek       0
order_purchase_month           0
approved_days                  0
dispatch_days                  0
delivery_days                  0
expected_delivery_days         0
delay_days                     0
delay_days_int                 0
is_delayed                     0
delay_days_cat                 0
main_category                  0
sub_catego

### 위도 경도 활용해서 거리 파생변수 생성

In [59]:
# Haversine 거리 계산 함수 (지구 곡률을 반영한 두 좌표 간의 직선 거리 km)
def haversine_vectorize(lat1, lon1, lat2, lon2):
    # 라디안 변환
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c # 6371: 지구의 평균 반지름(km)
    return km

# 거리(distance_km) 파생 변수 생성
df['distance_km'] = haversine_vectorize(
    df['customer_lat'], df['customer_lng'],
    df['seller_lat'], df['seller_lng']
)

#### 거리 파생변수 결측치 처리

In [60]:
print(f"distance_km 결측치 처리 전 행 수 : {len(df)}") # 95784
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median()) # 거리 중앙값으로 대체
print(f"distance_km 결측치 처리 후 행 수 : {len(df)}") # 95784

# 모델 학습에 불필요한 중간 컬럼 삭제
cols_to_drop = ['customer_zip_code_prefix', 'seller_zip_code_prefix', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng']
df = df.drop(columns=cols_to_drop)

distance_km 결측치 처리 전 행 수 : 95784
distance_km 결측치 처리 후 행 수 : 95784


In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95784 entries, 0 to 95783
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  95784 non-null  str    
 1   customer_id               95784 non-null  str    
 2   customer_unique_id        95784 non-null  str    
 3   customer_city             95784 non-null  str    
 4   customer_state            95784 non-null  str    
 5   total_items_count         95784 non-null  int64  
 6   seller_id                 95784 non-null  str    
 7   price                     95784 non-null  float64
 8   freight_value             95784 non-null  float64
 9   review_score              95784 non-null  int64  
 10  category                  94414 non-null  str    
 11  seller_city               95784 non-null  str    
 12  seller_state              95784 non-null  str    
 13  order_purchase_dayofweek  95784 non-null  str    
 14  order_purchase_mo

In [62]:
df.isna().sum()

order_id                       0
customer_id                    0
customer_unique_id             0
customer_city                  0
customer_state                 0
total_items_count              0
seller_id                      0
price                          0
freight_value                  0
review_score                   0
category                    1370
seller_city                    0
seller_state                   0
order_purchase_dayofweek       0
order_purchase_month           0
approved_days                  0
dispatch_days                  0
delivery_days                  0
expected_delivery_days         0
delay_days                     0
delay_days_int                 0
is_delayed                     0
delay_days_cat                 0
main_category                  0
sub_category                   0
distance_km                    0
dtype: int64

### 머신러닝 모델 생성시 절대 안쓸만한 컬럼 제거

In [63]:
cols_to_drop = ['order_id','customer_id','customer_city','customer_state','seller_id',
                'category','seller_city','seller_state','delay_days_int']
df = df.drop(columns=cols_to_drop)
df['delay_days'] = df['delay_days'].astype(int) # delay_days 정수화

In [64]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95784 entries, 0 to 95783
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_unique_id        95784 non-null  str    
 1   total_items_count         95784 non-null  int64  
 2   price                     95784 non-null  float64
 3   freight_value             95784 non-null  float64
 4   review_score              95784 non-null  int64  
 5   order_purchase_dayofweek  95784 non-null  str    
 6   order_purchase_month      95784 non-null  int64  
 7   approved_days             95784 non-null  int64  
 8   dispatch_days             95784 non-null  int64  
 9   delivery_days             95784 non-null  int64  
 10  expected_delivery_days    95784 non-null  int64  
 11  delay_days                95784 non-null  int64  
 12  is_delayed                95784 non-null  int64  
 13  delay_days_cat            95784 non-null  str    
 14  main_category    

In [65]:
df.isna().sum()

customer_unique_id          0
total_items_count           0
price                       0
freight_value               0
review_score                0
order_purchase_dayofweek    0
order_purchase_month        0
approved_days               0
dispatch_days               0
delivery_days               0
expected_delivery_days      0
delay_days                  0
is_delayed                  0
delay_days_cat              0
main_category               0
sub_category                0
distance_km                 0
dtype: int64

In [66]:
df.to_csv('./data/ml_data.csv')

In [67]:
# feature engineering? 까짓거 한번 해보죠

df_ml = df.copy()

# 베송속도 관련 파생피처들
df_ml['delivery_speed'] = df_ml['distance_km'] / df_ml['delivery_days']
df_ml['day_per_km'] = df_ml['delivery_days'] / df_ml['distance_km']
df_ml['delivery_ratio'] = df_ml['delivery_days'] / df_ml['expected_delivery_days']

# 가격 대비 배송비 & 상품당 배송비
df_ml['freight_ratio'] = df_ml['freight_value'] / df_ml['price']
df_ml['freight_per_item'] = df_ml['freight_value'] / df_ml['total_items_count']


# 거리 구간화
def distance_group(km):
    if km < 100:
        return 0
    elif km < 400:
        return 1
    elif km < 700:
        return 2
    elif km < 1000:
        return 3
    else:
        return 4

df_ml['distance_cat'] = df_ml['distance_km'].apply(distance_group).astype('int')



# delivery_days * distance_km
df_ml["delivery_distance"] = (df_ml["delivery_days"].fillna(0) * df_ml["distance_km"].fillna(0))

# delivery_days * price
df_ml["delivery_price"] = (df_ml["delivery_days"] * df_ml["price"])


df_ml = df_ml.replace([np.inf, -np.inf], np.nan)
num_cols = df_ml.select_dtypes(include=[np.number]).columns
df_ml[num_cols] = df_ml[num_cols].fillna(0)


In [68]:
# 회귀모델 시
X = df_ml.drop(columns=['review_score','customer_unique_id'])
y = df_ml['review_score']

In [69]:
X = pd.get_dummies(X, drop_first=True)

# Train+Valid / Test 분리
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train / Valid 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid,
    test_size=0.2,
    random_state=42,
    stratify=y_train_valid
)


In [70]:
# LGBM

model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)

model.fit(X_train, y_train)

# 6) valid 평가
valid_pred = model.predict(X_valid)
valid_r2 = r2_score(y_valid, valid_pred)
valid_mae = mean_absolute_error(y_valid, valid_pred)
valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))

print("R2  :", valid_r2)
print("MAE :", valid_mae)
print("RMSE:", valid_rmse)

# 7) test 평가
test_pred = model.predict(X_test)
test_r2 = r2_score(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("R2  :", test_r2)
print("MAE :", test_mae)
print("RMSE:", test_rmse)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003499 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3025
[LightGBM] [Info] Number of data points in the train set: 61301, number of used features: 55
[LightGBM] [Info] Start training from score 4.155821
R2  : 0.21334061415284
MAE : 0.8713751823251886
RMSE: 1.1386917862339283
R2  : 0.22090094853123843
MAE : 0.8657946802133892
RMSE: 1.1330674654576167


In [71]:
imp = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(imp.head(20))

delivery_ratio                        1141
freight_ratio                         1092
price                                 1069
freight_value                         1024
delivery_price                        1019
freight_per_item                       991
distance_km                            973
delivery_distance                      942
delivery_speed                         859
delay_days                             823
expected_delivery_days                 775
dispatch_days                          736
order_purchase_month                   703
delivery_days                          582
day_per_km                             364
total_items_count                      255
approved_days                          203
order_purchase_dayofweek_sunday         81
main_category_Home & Living             76
order_purchase_dayofweek_wednesday      75
dtype: int32
